# WUS station density 

Computes the manuscript text stat *"1 station per ~1,200 km² in mountain regions of the Western U.S."*:
aggregate station count, total mountain-range area (GMBA v2.0, ranges containing at least one station,
Albers equal-area), area per station, and stations per 100 km², limited to the Western US.

In [ ]:
from pathlib import Path

import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

from global_snowmelt_runoff_onset.results import save_result_table

In [ ]:
# Same version-scoped inputs as 4_snow_pillow_representativeness.ipynb -- set VERSION
# to match the config used in 1_create_snow_pillow_comparison_dataset.ipynb.
VERSION = 'v10'
COMPARISON_DATA_DIR = Path('data/comparison_datasets') / VERSION

In [ ]:
# our snow pillow station locations (the sites used in the comparison) -- identical
# source to 4_snow_pillow_representativeness.ipynb
max_swe_timing_ds = xr.open_zarr(COMPARISON_DATA_DIR / 'max_snow_pillow_swe_timing.zarr')
max_swe_timing_ds

In [ ]:
all_stations_gdf = gpd.GeoDataFrame(
    {'station_id': max_swe_timing_ds['station_id'].values},
    geometry=gpd.points_from_xy(max_swe_timing_ds['longitude'].values,
                                max_swe_timing_ds['latitude'].values),
    crs='EPSG:4326',
)
all_stations_gdf

In [ ]:
url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
gmba_gdf = gpd.read_file("zip+" + url)
gmba_gdf

In [ ]:
# only keep gmba ranges with stations, do a spatial join
gmba_with_stations_gdf = gpd.sjoin(gmba_gdf, all_stations_gdf, how="inner", predicate='intersects')
gmba_names = gmba_with_stations_gdf['MapName'].unique()
gmba_with_stations_gdf = gmba_gdf[gmba_gdf['MapName'].isin(gmba_names)]
gmba_with_stations_gdf

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
gmba_with_stations_gdf.plot(ax=ax, color='lightgrey', edgecolor='black')
all_stations_gdf.plot(ax=ax, color='red', markersize=5)

In [ ]:
# limit to the Western US: the shared inventory also has BC / Norway / Nepal stations,
# so bound by the WUS box (same rect the GMBA ranges are clipped to below), not just < 49N
WUS_stations_gdf = all_stations_gdf.cx[-125:-66, 24:49]
WUS_stations_gdf

In [ ]:
WUS_gmba_with_stations_gdf = gmba_with_stations_gdf.clip_by_rect(-125,24,-66,49)
WUS_gmba_with_stations_gdf

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
WUS_gmba_with_stations_gdf.plot(ax=ax, color='lightgrey', edgecolor='black')
WUS_stations_gdf.plot(ax=ax, color='red', markersize=5)

In [ ]:
# calculate total area
WUS_gmba_with_stations_aea_gdf = WUS_gmba_with_stations_gdf.to_crs(epsg=5070)  # Albers Equal Area# calculate total area
WUS_gmba_with_stations_aea_gdf

In [ ]:
num_stations = len(WUS_stations_gdf)
num_stations

In [ ]:
total_area_km2 = WUS_gmba_with_stations_aea_gdf.area.sum()/1e6
total_area_km2

In [ ]:
area_per_station_km2 = total_area_km2 / num_stations
area_per_station_km2

In [ ]:
# calculate station density per 100 km^2
station_density_per_100km2 = num_stations / (total_area_km2 / 100)
station_density_per_100km2

In [ ]:
summary_df = pd.DataFrame([{
    'n_stations': num_stations,
    'total_mountain_area_km2': round(total_area_km2, 1),
    'area_per_station_km2': round(area_per_station_km2, 1),
    'stations_per_100km2': round(station_density_per_100km2, 4),
    'station_source': f'max_snow_pillow_swe_timing.zarr ({VERSION}) usable stations, WUS box -125..-66E / 24..49N',
    'range_source': 'GMBA Inventory v2.0 standard, ranges containing >=1 station, Albers EPSG:5070 areas',
}])
save_result_table(summary_df, 'station_density', version=VERSION)
print(f"1 station per ~{area_per_station_km2:,.0f} km^2 of WUS mountain area (manuscript Sect. 5.1)")
summary_df